In [1]:
from preprocessing import (
    load_dataset,
    create_features,
    select_features,
    encode_features,
    split_data,
)
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

import joblib

In [2]:
df = load_dataset("../data/historical/flight_delays.csv")

In [3]:
df.head()

,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance
0,1,United,4558,ORD,MIA,2024-09-01 08:11,2024-09-01 08:30,2024-09-01 12:11,2024-09-01 12:19,8,Weather,True,False,Boeing 737,N71066,1031
1,2,Delta,8021,LAX,MIA,2024-09-01 10:25,2024-09-01 10:41,2024-09-01 13:25,2024-09-01 13:27,2,Air Traffic Control,True,True,Airbus A320,N22657,1006
2,3,Southwest,7520,DFW,SFO,2024-09-01 16:53,2024-09-01 17:05,2024-09-01 17:53,2024-09-01 18:07,14,Weather,True,True,Boeing 737,N95611,2980
3,4,Delta,2046,ORD,BOS,2024-09-01 14:44,2024-09-01 15:04,2024-09-01 18:44,2024-09-01 18:34,-10,NaN,False,False,Boeing 777,N90029,1408
4,5,Delta,6049,LAX,SEA,2024-09-01 01:51,2024-09-01 02:08,2024-09-01 05:51,2024-09-01 06:15,24,Air Traffic Control,False,True,Boeing 737,N27417,2298


In [4]:
df = create_features(df)

In [5]:
df.head()

,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance,DepartureHour,Month,DayOfWeek,Delayed
0,1,United,4558,ORD,MIA,2024-09-01 08:11:00,2024-09-01 08:30,2024-09-01 12:11,2024-09-01 12:19,8,Weather,True,False,Boeing 737,N71066,1031,8,9,6,0
1,2,Delta,8021,LAX,MIA,2024-09-01 10:25:00,2024-09-01 10:41,2024-09-01 13:25,2024-09-01 13:27,2,Air Traffic Control,True,True,Airbus A320,N22657,1006,10,9,6,0
2,3,Southwest,7520,DFW,SFO,2024-09-01 16:53:00,2024-09-01 17:05,2024-09-01 17:53,2024-09-01 18:07,14,Weather,True,True,Boeing 737,N95611,2980,16,9,6,0
3,4,Delta,2046,ORD,BOS,2024-09-01 14:44:00,2024-09-01 15:04,2024-09-01 18:44,2024-09-01 18:34,-10,NaN,False,False,Boeing 777,N90029,1408,14,9,6,0
4,5,Delta,6049,LAX,SEA,2024-09-01 01:51:00,2024-09-01 02:08,2024-09-01 05:51,2024-09-01 06:15,24,Air Traffic Control,False,True,Boeing 737,N27417,2298,1,9,6,1


In [6]:
df = select_features(df)

In [7]:
df.head()

,Airline,Origin,Destination,DepartureHour,Month,DayOfWeek,Distance,AircraftType,Delayed
0,United,ORD,MIA,8,9,6,1031,Boeing 737,0
1,Delta,LAX,MIA,10,9,6,1006,Airbus A320,0
2,Southwest,DFW,SFO,16,9,6,2980,Boeing 737,0
3,Delta,ORD,BOS,14,9,6,1408,Boeing 777,0
4,Delta,LAX,SEA,1,9,6,2298,Boeing 737,1


In [8]:
df, airline_encoder, origin_encoder, destination_encoder, aircraft_encoder = encode_features(df)

In [9]:
df.head()

,Airline,Origin,Destination,DepartureHour,Month,DayOfWeek,Distance,AircraftType,Delayed
0,3,4,2,8,9,6,1031,1,0
1,1,3,2,10,9,6,1006,0,0
2,2,1,4,16,9,6,2980,1,0
3,1,4,0,14,9,6,1408,2,0
4,1,3,3,1,9,6,2298,1,1


In [10]:
joblib.dump(airline_encoder, "../models/airline_encoder.pkl")
joblib.dump(origin_encoder, "../models/origin_encoder.pkl")
joblib.dump(destination_encoder, "../models/destination_encoder.pkl")
joblib.dump(aircraft_encoder, "../models/aircraft_encoder.pkl")

['../models/aircraft_encoder.pkl']

In [11]:
loaded_airline_encoder = joblib.load("../models/airline_encoder.pkl")

print(loaded_airline_encoder.classes_)

['American Airlines' 'Delta' 'Southwest' 'United']


In [12]:
loaded_origin_encoder = joblib.load("../models/origin_encoder.pkl")

print(loaded_origin_encoder.classes_)

['ATL' 'DFW' 'JFK' 'LAX' 'ORD']


In [13]:
loaded_destination_encoder = joblib.load("../models/destination_encoder.pkl")

print(loaded_destination_encoder.classes_)

['BOS' 'JFK' 'MIA' 'SEA' 'SFO']


In [14]:
loaded_aircraft_encoder = joblib.load("../models/aircraft_encoder.pkl")

print(loaded_aircraft_encoder.classes_)

['Airbus A320' 'Boeing 737' 'Boeing 777']


In [15]:
X_train, X_test, y_train, y_test = split_data(df)

In [16]:
print(X_train.shape)
print(X_test.shape)

(1398101, 8)
(349526, 8)


In [17]:
model = RandomForestClassifier(
    n_estimators=10,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

In [18]:
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, n_estimators=10, n_jobs=-1, random_state=42)

In [19]:
y_pred = model.predict(X_test)

In [20]:
y_pred[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [21]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6112


In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      1.00      0.76    213615
           1       0.67      0.00      0.00    135911

    accuracy                           0.61    349526
   macro avg       0.64      0.50      0.38    349526
weighted avg       0.63      0.61      0.46    349526



In [23]:
print(confusion_matrix(y_test, y_pred))

[[213614      1]
 [135909      2]]


In [25]:
joblib.dump(model, "../models/model.pkl")

['../models/model.pkl']

In [26]:
loaded_model = joblib.load("../models/model.pkl")

print(type(loaded_model))
print(loaded_model)

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
RandomForestClassifier(max_depth=8, n_estimators=10, n_jobs=-1, random_state=42)
